<a href="https://colab.research.google.com/github/kasturikirankumar1101-lab/AI_TOOLS/blob/main/AirlinesChatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
import mysql.connector as mq

# Loading environment variables from .ENV file and checking whether the OpenAI Key is available in the .ENV file.
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

# Systemprompt to handle the flights prices update and get the details of the flights based on the destination to share with customers
system_message = """
You are an intelligent Flight Price Assistant.

Your role is to help users:
1. Check flight ticket prices
2. Update/store new flight price information in the database

You have access to the following tools:
- A tool to GET flight prices from the database
- A tool to UPDATE/STORE flight price details in the database

----------------------------------------
CORE BEHAVIOR:
----------------------------------------

1. Understand User Intent:
- If the user is asking for flight prices → use the GET tool
- If the user wants to add/update flight price → use the UPDATE tool

2. Collect Required Information:
Before calling any tool, ensure you have ALL required details.

For flight price lookup:
- Source city
- Destination city
- Travel date

For updating flight price:
- Source city
- Destination city
- Travel date
- Price
- Airline (if applicable)

If any detail is missing:
→ Ask the user clearly for the missing information
→ DO NOT call the tool until all required inputs are available

----------------------------------------
TOOL USAGE RULES:
----------------------------------------

- Always use tools for:
  ✔ Fetching flight prices
  ✔ Updating/storing flight data

- NEVER guess or fabricate flight prices
- NEVER answer from memory when real data is required

- When calling a tool:
  → Pass structured and accurate arguments
  → Ensure values are clean and validated

----------------------------------------
RESPONSE FLOW:
----------------------------------------

1. If information is missing:
   → Ask follow-up questions

2. If ready:
   → Call the appropriate tool

3. After tool execution:
   → Present the result clearly to the user

----------------------------------------
RESPONSE STYLE:
----------------------------------------

- Be clear, concise, and helpful
- Use a professional and friendly tone
- Format outputs neatly

Example:
"✈️ Flight from Hyderabad to Delhi on 25th March costs ₹5,200."

----------------------------------------
ERROR HANDLING:
----------------------------------------

- If no data is found:
  → Inform the user politely and suggest alternatives

- If update is successful:
  → Confirm clearly:
    "✅ Flight price updated successfully."

----------------------------------------
IMPORTANT:
----------------------------------------

- Do NOT expose tool names or internal logic to the user
- Do NOT hallucinate data
- Always prefer tool execution over assumptions
"""

DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("""
CREATE TABLE IF NOT EXISTS airlines_prices (
    source TEXT,
    destination TEXT,
    date TEXT,
    airline TEXT,
    price REAL,
    PRIMARY KEY (source, destination, date, airline)
    )
     """)
    conn.commit()

def get_ticket_price(source, destination,date):
    print(f"DATABASE TOOL CALLED: Getting price for {source} to {destination}", flush=True)
    with mq.connect(host="localhost", user="root", passwd="sainath",database="flight_db") as conn:
        cursor = conn.cursor()
        SQLquery = '''SELECT  airlines_name, price
                        FROM prices
                        WHERE source = %s AND destination = %s AND date = %s'''
        cursor.execute(SQLquery, (source.lower(), destination.lower(), date))
        result = cursor.fetchone()
        print(f"what is result[0] = {result[1]}")
        airline = result[0]
        price = result[1]
        return f"Ticket price from {source}to {destination} with {airline} is ${price}" if result else "No price data available for this city"



def set_ticket_price(source, destination,date,airline, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("""
                        INSERT INTO airlines_prices (source, destination, date, airline, price)
                                VALUES (?, ?, ?, ?, ?)
                                ON CONFLICT(source, destination, date, airline)
                                DO UPDATE SET price = excluded.price
                        """, (source.lower(), destination.lower(), date, airline.lower(), price))
        conn.commit()
    print("set_price called")


price_get_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "source_city": {
                "type": "string",
                "description": "The city that the customer wants to travel from",
            },
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "date": {
                "type": "string",
                "description": "the date at which customer wants to travel",
            },

        },
        "required": ["source_city", "destination_city", "date"],
        "additionalProperties": False
    }
}

price_set_function = {
    "name": "set_ticket_price",
    "description": "Set the ticket price of a flight ticket to the city.",
    "parameters": {
        "type": "object",
        "properties": {
            "source_city": {
                "type": "string",
                "description": "The city that the customer wants to travel from",
            },
            "destination_city": {
                "type": "string",
                "description": "The ticket price that the user wants to set for the city",
            },
            "date": {
                "type": "string",
                "description": "the date at which the flight available",
            },
            "airline_name": {
                "type": "string",
                "description": "The name of the airlines",
            },
            "destination_price": {
                "type": "number",
                "description": "The ticket price that the user wants to set for the city",
            },

        },
        "required": ["source_city", "destination_city", "date", "airline_name", "destination_price"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_get_function}, {"type": "function", "function": price_set_function}]



def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content




def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    if name == "get_ticket_price":

        source = arguments.get('source_city')
        destination = arguments.get('destination_city')
        date = arguments.get('date')
        price = ''
        airline = get_ticket_price(source,destination,date)
        response = {
            "role": "tool",
            "content": json.dumps({
                "source": source,
                "destination": destination,
                "date": date,
                "airline": airline,
                "price": price
            }),
            "tool_call_id": tool_call.id
        }

    elif name == "set_ticket_price":
      source = arguments.get('source_city')
      destination = arguments.get('destination_city')
      date = arguments.get('date')
      airline_name=  arguments.get('airline_name')
      price =  arguments.get('destination_price')
      set_ticket_price(source, destination,date,airline_name,price)
      print("set_price completed")
      response = {
            "role": "tool",
            "content": "price has been updated in DB",
            "tool_call_id": tool_call.id
        }


    return response


gr.ChatInterface(fn=chat, type="messages", fill_height=True).launch(inbrowser=True)